# SoundAQnet — Python API Tutorial

This notebook walks through the full Python API:

1. [Installation](#1-installation)
2. [Feature extraction — single file](#2-feature-extraction--single-file)
3. [Feature extraction — multiple files with parallelism](#3-feature-extraction--multiple-files)
4. [Inference — single sample](#4-inference--single-sample)
5. [Inference — batch from .npy directories](#5-inference--batch-from-npy-directories)
6. [End-to-end — audio files → predictions](#6-end-to-end--audio-files--predictions)
7. [Reading and interpreting results](#7-reading-and-interpreting-results)
8. [Choosing bundled models](#8-choosing-bundled-models)
9. [EmoSoundscape — valence/arousal](#9-emosoundscape--valencearousal)
10. [Summary](#10-summary)

---

> **Platform note** — Loudness extraction uses the bundled `ISO_532-1.exe` on **Windows** and the pure-Python [mosqito](https://github.com/Eomys/MoSQITo) library on **macOS / Linux**.


## 1 · Installation

For GPU acceleration, install PyTorch **before** `soundaqnet` so the CUDA/CPU variant is resolved correctly. CPU-only installs can use PyPI defaults.

```bash
# CPU-only or Apple Silicon / MPS
pip install torch torchvision torchaudio

# NVIDIA GPU — CUDA 12.1
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Then install soundaqnet
pip install soundaqnet
```

Verify the install:


In [ ]:
import soundaqnet
print("soundaqnet version:", soundaqnet.__version__)

import torch
print("torch version:     ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())

---
## 2 · Feature extraction — single file

Two feature types are required by the model:

| Feature | Function | Output shape | Description |
|---|---|---|---|
| Log-mel spectrogram | `extract_mel_from_file` | `(T, 64)` | STFT + mel filterbank at 16 kHz |
| ISO 532-1 loudness | `extract_loudness_from_file` | `(T, 1)` | Time-varying Zwicker loudness in sone, calibrated from the bundled 1 kHz / 60 dB SPL reference |


In [ ]:
from soundaqnet.feature_extraction import extract_mel_from_file, extract_loudness_from_file

# Replace with any mono audio file (.wav, .mp3, .flac, .ogg, .aiff, .m4a, .opus)
AUDIO_FILE = "my_clip.wav"

mel  = extract_mel_from_file(AUDIO_FILE)
loud = extract_loudness_from_file(AUDIO_FILE)

print(f"mel shape:      {mel.shape}   dtype: {mel.dtype}")
print(f"loudness shape: {loud.shape}  dtype: {loud.dtype}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

axes[0].imshow(mel.T, origin="lower", aspect="auto", cmap="viridis")
axes[0].set_title("Log-mel spectrogram  (64 mel bins)")
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("Mel bin")

time_axis = np.arange(len(loud)) / 500.0   # ISO 532-1 uses 2 ms steps → 500 Hz
axes[1].plot(time_axis, loud[:, 0])
axes[1].set_title("Time-varying loudness  (ISO 532-1 Zwicker, sone)")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Loudness (sone)")

plt.tight_layout()
plt.show()

### Saving features to disk

Pass `output_dir` to write `.npy` files that can be used later with `SoundAQnet.predict()`.


In [ ]:
mel_saved  = extract_mel_from_file(AUDIO_FILE,  output_dir="mel_features/")
loud_saved = extract_loudness_from_file(AUDIO_FILE, output_dir="loudness_features/")

# mel_features/my_clip.npy and loudness_features/my_clip.npy are now on disk
import pathlib
print("Saved files:", list(pathlib.Path("mel_features").glob("*.npy")))

---
## 3 · Feature extraction — multiple files

Pass a **list** of paths to process several clips at once.
Both functions return a `dict[stem → array]` and accept `num_workers` for parallel extraction. On macOS/Linux, loudness uses processes because mosqito has Python-level loops; on Windows, loudness uses threads around the bundled executable.


In [ ]:
AUDIO_FILES = ["clip_A.wav", "clip_B.wav", "clip_C.wav"]  # replace with real paths

# ── Parallel mel extraction (4 threads) ──────────────────────────────────────
mels = extract_mel_from_file(
    AUDIO_FILES,
    output_dir="mel_features/",
    num_workers=4,
)
print("Mel shapes:", {k: v.shape for k, v in mels.items()})

# ── Parallel loudness extraction (4 threads) ─────────────────────────────────
louds = extract_loudness_from_file(
    AUDIO_FILES,
    output_dir="loudness_features/",
    num_workers=4,
)
print("Loudness shapes:", {k: v.shape for k, v in louds.items()})

### Batch extraction from a whole directory

Use `extract_mel` / `extract_loudness` when you have a folder of audio files.
Files whose `.npy` output already exists are automatically skipped. `extract_loudness` also accepts `start_idx` / `end_idx` for splitting large sorted file lists across jobs.


In [ ]:
from soundaqnet.feature_extraction import extract_mel, extract_loudness

AUDIO_DIR    = "audio/"            # directory of audio clips
MEL_DIR      = "mel_features/"
LOUDNESS_DIR = "loudness_features/"

# Both functions are parallel-safe and skip completed files automatically.
extract_mel(AUDIO_DIR, MEL_DIR, num_workers=4)
extract_loudness(AUDIO_DIR, LOUDNESS_DIR, num_workers=4)

---
## 4 · Inference — single sample

`predict_sample(mel, loudness)` runs the model on a **single** pair of numpy arrays.


In [ ]:
from soundaqnet import SoundAQnet

# Load the default bundled model (auto-selects CUDA → MPS → CPU)
model = SoundAQnet()
print(model)

In [ ]:
# Reuse mel / loud from Section 2
result = model.predict_sample(mel, loud)

print("=== Single-clip prediction ===")
print(f"  Scene:           {result['scene']}")
print(f"  ISO Pleasantness: {result['isop']:+.3f}")
print(f"  ISO Eventfulness: {result['isoe']:+.3f}")
print()
print("  PAQ 8-D scores:")
for dim in ["pleasant", "eventful", "chaotic", "vibrant",
            "uneventful", "calm", "annoying", "monotonous"]:
    print(f"    {dim:<12}: {result[dim]:+.3f}")
print()
print("  Top-5 events:")
for ev in result["top_events"]:
    prob = result["event_probs"][ev]
    print(f"    {ev:<30} {prob:.3f}")

---
## 5 · Inference — batch from .npy directories

`predict(mel_dir, loudness_dir)` reads all pre-extracted `.npy` files from disk and
returns a `pandas.DataFrame` — one row per clip.  
Adjust `batch_size` to trade memory for speed.


In [ ]:
# Extract to disk first (Section 3 above already did this, so it will skip)
extract_mel(AUDIO_DIR, MEL_DIR, num_workers=4)
extract_loudness(AUDIO_DIR, LOUDNESS_DIR, num_workers=4)

# Run inference on all clips in the directories
df = model.predict(
    mel_dir=MEL_DIR,
    loudness_dir=LOUDNESS_DIR,
    batch_size=32,       # increase to 64-128 for GPU
)
print(df.shape)
df.head()

In [ ]:
# Inspect the columns
print(df.dtypes)
print()

# The event_probs column contains dicts — expand them if needed
event_df = df["event_probs"].apply(lambda d: pd.Series(d))
event_df.head()

In [ ]:
import pandas as pd

# Quick summary of affective quality dimensions
paq_cols = ["pleasant", "eventful", "chaotic", "vibrant",
            "uneventful", "calm", "annoying", "monotonous"]
print(df[paq_cols].describe().round(3))

### Circumplex plot of ISO Pleasantness vs Eventfulness


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df["isoe"], df["isop"], alpha=0.7)
for _, row in df.iterrows():
    ax.annotate(row["clip_id"], (row["isoe"], row["isop"]),
                fontsize=7, ha="center", va="bottom")
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0, color="k", linewidth=0.5)
ax.set_xlabel("ISO Eventfulness")
ax.set_ylabel("ISO Pleasantness")
ax.set_title("Soundscape circumplex")
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
plt.tight_layout()
plt.show()

---
## 6 · End-to-end — audio files → predictions

`predict_from_audio` extracts features internally and returns predictions directly.
No intermediate `.npy` files are written. For large datasets or resumable jobs, pre-extract features with `extract_mel` / `extract_loudness`, then call `predict`.


In [ ]:
# From a directory: rich DataFrame with top_events lists and event_probs dicts
# You can also pass output_csv="soundAQ.csv" and flat=True to write/return
# the CSV-ready table directly from model.predict_from_audio(...).
df_e2e = model.predict_from_audio(
    audio_dir=AUDIO_DIR,
    batch_size=32,
    num_workers=4,
)
df_e2e[["clip_id", "scene", "isop", "isoe"]].head()

In [ ]:
# Convert an existing rich prediction DataFrame to one complete CSV table
from soundaqnet import flatten_prediction_results

flat_df = flatten_prediction_results(df_e2e)
flat_df.to_csv("soundAQ.csv", index=False)
flat_df.head()


In [ ]:
# Or pass an explicit file list
df_files = model.predict_from_audio(
    audio_files=["clip_A.wav", "clip_B.wav"],
    batch_size=16,
    num_workers=2,
)
df_files

---
## 7 · Reading and interpreting results

The default prediction DataFrame is rich and Python-native:

| Column | Type | Description |
|---|---|---|
| `clip_id` | str | File stem (no extension); `predict_sample` uses `sample` |
| `scene` | str | Predicted acoustic scene: `'urban'`, `'suburban'`, or `'park'` |
| `isop` | float | ISO Pleasantness –1 … +1 |
| `isoe` | float | ISO Eventfulness –1 … +1 |
| `pleasant` … `monotonous` | float | PAQ 8-D affective quality scores |
| `top_events` | list[str] | Top-5 audio event labels (highest probability first) |
| `event_probs` | dict | Probability for each of the 15 audio event classes |

For CSV output, use `flatten_prediction_results(df)` or pass `flat=True` / `output_csv="soundAQ.csv"` to `predict(...)` or `predict_from_audio(...)`. The flat table expands all event probabilities into scalar columns such as `event_prob_Bird`.

In [ ]:
# Save results to CSV
df.drop(columns=["event_probs"]).to_csv("soundaqnet_results.csv", index=False)

# Or keep the full dict column and save to JSON
df.to_json("soundaqnet_results.json", orient="records", indent=2)

In [ ]:
# All 15 event class labels
from soundaqnet.inference import EVENT_LABELS
print(EVENT_LABELS)

---
## 8 · Choosing bundled models

The package bundles checkpoints for two related workflows.

### SoundAQnet

These models predict acoustic scene, event probabilities, ISO Pleasantness/Eventfulness, and PAQ 8-D affective quality.

| Name | ASC acc | AEC acc | PAQ F1 |
|---|---|---|---|
| `SoundAQnet_ASC96_AEC94_PAQ1027` **(default)** | 96% | 94% | 10.27 |
| `SoundAQnet_ASC96_AEC94_PAQ1039` | 96% | 94% | 10.39 |
| `SoundAQnet_ASC96_AEC94_PAQ1041` | 96% | 94% | 10.41 |
| `SoundAQnet_ASC96_AEC95_PAQ1052` | 96% | 95% | 10.52 |

### EmoSoundscape

This model predicts valence and arousal from package-native 122-feature audio summaries.

| Name | Input | 10-fold CV R2 |
|---|---|---|
| `EmoS_gradient_boosting` **(default)** | 122 package-native audio features | mean 0.796, valence 0.696, arousal 0.895 |


In [ ]:
from soundaqnet import SoundAQnet, EmoSoundscape
from soundaqnet.inference import BUNDLED_MODELS
from soundaqnet.emosoundscape import DEFAULT_EMOSOUNDSCAPE_MODEL

print("SoundAQnet models:", BUNDLED_MODELS)
print("EmoSoundscape model:", DEFAULT_EMOSOUNDSCAPE_MODEL)

# Load specific bundled models
model_v2 = SoundAQnet("SoundAQnet_ASC96_AEC95_PAQ1052")
emo_model = EmoSoundscape("EmoS_gradient_boosting")
print(model_v2)
print(emo_model)

# Or load from an absolute path
# model_custom = SoundAQnet("/path/to/my_finetuned.pth")
# emo_custom = EmoSoundscape("/path/to/my_model.pkl")


In [ ]:
# Force a specific device
model_cpu = SoundAQnet(device="cpu")
model_gpu = SoundAQnet(device="cuda")   # raises if CUDA not available
# model_mps = SoundAQnet(device="mps")  # Apple Silicon

---
## 9 · EmoSoundscape — valence/arousal

`EmoSoundscape` is a separate interface for predicting valence and arousal from audio. It uses the bundled `EmoS_gradient_boosting` model by default.


In [ ]:
from soundaqnet import EmoSoundscape
from soundaqnet.feature_extraction import extract_emosoundscape_features_from_file

emo = EmoSoundscape()
print(emo)

# Direct audio inference
emo_df = emo.predict_from_audio(audio_files=[AUDIO_FILE])
emo_df


In [ ]:
# Optional: inspect the package-native 122-feature vector
emo_features = extract_emosoundscape_features_from_file(AUDIO_FILE)
print(emo_features.shape)

emo_result = emo.predict_sample(emo_features, clip_id="my_clip")
emo_result


In [ ]:
# Directory inference
emo_batch = emo.predict_from_audio(audio_dir=AUDIO_DIR, batch_size=32)
emo_batch[["clip_id", "valence", "arousal"]].head()


---
## 10 · Summary

```python
from soundaqnet import SoundAQnet, EmoSoundscape, flatten_prediction_results
from soundaqnet.feature_extraction import (
    extract_mel_from_file, extract_loudness_from_file,
    extract_mel, extract_loudness,
)

# SoundAQnet: scene, events, ISO, PAQ
mel  = extract_mel_from_file("clip.wav")
loud = extract_loudness_from_file("clip.wav")
model = SoundAQnet()
result = model.predict_sample(mel, loud)
df = model.predict("mel/", "loudness/")
flat_df = model.predict("mel/", "loudness/", flat=True, output_csv="soundAQ.csv")
df_audio = model.predict_from_audio("audio/")
flat_audio = flatten_prediction_results(df_audio)

# EmoSoundscape: valence, arousal
emo = EmoSoundscape()
emo_df = emo.predict_from_audio(audio_files=["clip.wav"])
```